In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import HTML
import xgboost as xgb
from category_encoders import MEstimateEncoder

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression

plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

set_config(transform_output='pandas')

train = pd.read_csv('train.csv', index_col='Id')
test = pd.read_csv('test.csv', index_col='Id')

In [ ]:
data = train.copy()
# Creating new features used for outlier detection
data['TotalLivArea'] = data['TotalBsmtSF'] + data['GrLivArea']
data['TotalPorchArea'] = (data['WoodDeckSF'] + data['OpenPorchSF'] + data['EnclosedPorch'] +
                          data['3SsnPorch'] + data['ScreenPorch'])

# Identifying outlier indeces based on various features
outliers_index = data.loc[data['TotalBsmtSF'] > 2200].index.union(
    data.loc[data['SalePrice'] < 50000].index).union(
        data.loc[data['SalePrice'] > 450000].index).union(
            data.loc[data['TotalLivArea'] > 5200].index).union(
                data.loc[data['TotalLivArea'] < 1200].index).union(
                    data.loc[data['LotArea'] > 27000].index).union(
                        data.loc[data['TotalPorchArea'] > 1000].index).union(
                            data.loc[data['LotFrontage'] > 250].index).union(
                                data.loc[data['GarageArea'] > 970].index).union(
                                    data.loc[data['TotRmsAbvGrd'] > 10].index)
# data = data.drop(outliers_index)

X = data.drop('SalePrice', axis=1)
y = data.SalePrice

# data['OverallRating'] = data['OverallQual'] * data['OverallCond']
# data['log'] = np.sqrt(data['LotFrontage'])
# data['norm'] = (data['log'] - data['log'].mean())/data['log'].std()
# data['minmax'] = (data['OverallQual'] - data['OverallQual'].min())/(data['OverallQual'].max()-data['OverallQual'].min())
# sns.boxplot(x=data['LandSlope'], y=data['SalePrice'])
# sns.scatterplot(x=data['GarageFinish'], y=data['SalePrice'])
# HTML(data.sort_values(by='LotFrontage').head(10).to_html())
# data.loc[data['LotArea'].sort_values().tail(10).index, ['LotArea', 'norm']]

### Specific imputers
1. `BsmtExposure` and `BsmtFinType` values are missing in some cases, where there is a basement. Imputed them with mode, while observations with no basement are imputed accordingly.
2. Impute mean within categories on another feature.

In [ ]:
class BsmtDetailsReplacer(BaseEstimator, TransformerMixin):
    def __init__(self, BsmtCond):
        self.BsmtCond = BsmtCond
        self.y = None
    def fit(self, X, y=None):        
        return self
    def transform(self, X):
        for col in X.columns:
            X.loc[(self.BsmtCond.notna()) & (X[col].isna()), col] = X[col].mode()[0]
        return X

BsmtExposure_pl = Pipeline(steps=[
    ('impute', BsmtDetailsReplacer(X['BsmtCond'])),
    ('fill_na', SimpleImputer(strategy='constant', fill_value='NB'))
])
BsmtFinType_pl = Pipeline(steps=[
    ('impute', BsmtDetailsReplacer(X['BsmtCond'])),
    ('fill_na', SimpleImputer(strategy='constant', fill_value='No'))
])

class MeanPerCategoryImputer(BaseEstimator, TransformerMixin):
    def __init__(self, cat):
        self.cat = cat
        self.y = None
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        for col in X.columns:
            X.loc[X[col].isna(), col] = X.groupby(self.cat)[col].transform('mean')
        return X

In [ ]:
# Features to drop and impute with different strategies
drop_cols = ['Alley', 'Condition2', 'MoSold', 'YrSold', 'MiscFeature', 'Utilities', 'LowQualFinSF',
             '1stFlrSF', 'GrLivArea', 'Heating', 'PoolQC', 'PoolArea', 'MiscVal', 'Street',
             'Exterior2nd', 'BsmtUnfSF', 'KitchenAbvGr', 'RoofStyle', 'Condition1', 'LandSlope',
             'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
impute_zero_cols = ['TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt', 'MasVnrArea']
impute_No_cols = ['BsmtQual', 'BsmtCond', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond',
                  'Fence', 'GarageType']
impute_mean_cols = ['GarageCars', 'GarageArea']
impute_mode_cols = ['KitchenQual', 'Functional', 'Electrical']

impute_ct = ColumnTransformer(transformers=[
    ('BsmtExposure_prep', BsmtExposure_pl, ['BsmtExposure']),
    ('BsmtFinType_prep', BsmtFinType_pl, ['BsmtFinType1', 'BsmtFinType2']),
    ('impute_zero', SimpleImputer(strategy='constant', fill_value=0), impute_zero_cols),
    ('impute_No', SimpleImputer(strategy='constant', fill_value='No'), impute_No_cols),
    ('impute_mean', SimpleImputer(strategy='mean'), impute_mean_cols),
    ('impute_mean_per_LotConfig', MeanPerCategoryImputer(cat=X['LotConfig']), ['LotFrontage']),
    ('impute_mode', SimpleImputer(strategy='most_frequent'), impute_mode_cols),
    ('drop', 'drop', drop_cols)
], remainder='passthrough', verbose_feature_names_out=False)

#### 1. Merging uninformative categories
Rare or similar categories are merged to reduce the number of variables.
Variable `cat_map` used in `replace_cats` function allows to easily add sets of categories to replace for many features.

#### 2. Encoding features used in deriving new ones in the next step

In [ ]:
RoofMatl_other = ['ClyTile', 'Membran', 'Metal', 'Roll', 'Tar&Grv']
RoofMatl_wood = ['WdShake', 'WdShngl']
RoofMatl_cat_map = (dict(zip(RoofMatl_wood, ['Wood']*len(RoofMatl_wood))) |
                    dict(zip(RoofMatl_other, ['Other']*len(RoofMatl_other))))

Foundation_other = ['Stone', 'Wood', 'Slab']
Foundation_cat_map = dict(zip(Foundation_other, ['Other']*len(Foundation_other)))

BldgType_twnhs = ['TwnhsE']
BldgType_cat_map = dict(zip(BldgType_twnhs, ['Twnhs']*len(BldgType_twnhs)))

LotConfig_other = ['Corner', 'FR2', 'FR3']
LotConfig_cat_map = dict(zip(LotConfig_other, ['Other']*len(LotConfig_other)))

SaleType_WD = ['CWD', 'VWD']
SaleType_Oth = ['ConLw', 'ConLI', 'ConLD', 'Con']
SaleType_cat_map = (dict(zip(SaleType_WD, ['WD']*len(SaleType_WD))) |
                    dict(zip(SaleType_Oth, ['Oth']*len(SaleType_Oth))))

Exterior1st_Rare = ['BrkComm', 'Stone', 'AsphShn', 'ImStucc', 'CBlock']
Exterior1st_cat_map = dict(zip(Exterior1st_Rare, ['Rare']*len(Exterior1st_Rare)))

LandContour_Flat = ['Lvl', 'Low']
LandContour_cat_map = dict(zip(LandContour_Flat, ['Flat']*len(LandContour_Flat)))

GarageType_Rare = ['CarPort', '2Types', 'Basment']
GarageType_cat_map = dict(zip(GarageType_Rare, ['Rare']*len(GarageType_Rare)))

Electrical_cat_map = {'Mix':'FuseA'}

SaleCondition_Other = ['Abnorml', 'AdjLand', 'Alloca', 'Family']
SaleCondition_Normal = ['Alloca']
SaleCondition_cat_map = (dict(zip(SaleCondition_Other, ['Other']*len(SaleCondition_Other))) |
                         dict(zip(SaleCondition_Normal, ['Normal']*len(SaleCondition_Normal))))

LotShape_IR = ['IR1', 'IR2', 'IR3']
LotShape_cat_map = dict(zip(LotShape_IR, ['IR']*len(LotShape_IR)))

# Combining all the category replacements
replace_cats_cols = ['RoofMatl', 'Foundation', 'BldgType', 'LotConfig',
                     'SaleType', 'Exterior1st', 'LandContour', 'GarageType',
                     'Electrical', 'SaleCondition', 'LotShape']
cat_maps = [RoofMatl_cat_map, Foundation_cat_map, BldgType_cat_map, LotConfig_cat_map,
            SaleType_cat_map, Exterior1st_cat_map, LandContour_cat_map, GarageType_cat_map,
            Electrical_cat_map, SaleCondition_cat_map, LotShape_cat_map]
cat_map = dict(zip(replace_cats_cols, cat_maps))

def replace_cats(X):
    for col in cat_map.keys():        
        if col in X.columns:
            X[col] = X[col].replace(cat_map[col])
    return X

# Encoding selected features required for creating new ones in the next step. Other features are encoded later.
BsmtFinType_cats = ['No', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
ExterQC_cats = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
BsmtQC_cats = ['No', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

replace_cats_ct = ColumnTransformer(transformers=[
    ('replace_cats', FunctionTransformer(replace_cats), replace_cats_cols),
    ('encode_BsmtFinType',
        OrdinalEncoder(categories=[BsmtFinType_cats]*2, encoded_missing_value=-1),
        ['BsmtFinType1', 'BsmtFinType2']),
    ('encode_ExterQC',
        OrdinalEncoder(categories=[ExterQC_cats]*2, encoded_missing_value=-1),
        ['ExterQual', 'ExterCond']),
    ('encode_BsmtQC',
        OrdinalEncoder(categories=[BsmtQC_cats]*2, encoded_missing_value=-1),
        ['BsmtQual', 'BsmtCond']),
    ('encode_GarageQC',
        OrdinalEncoder(categories=[BsmtQC_cats]*2, encoded_missing_value=-1),
        ['GarageQual', 'GarageCond']),
], remainder='passthrough', verbose_feature_names_out=False)

### Adding new features
* `BsmtFinCombined`: Combining ordinally encoded `BsmtFinType` features with their respective areas `BsmtFinSF` to get a weighted mean value.
* `OpenPorchPerc`: Percentage of porch area without cover
* `sqrt_norm_GarageExtraArea`: Normalized linear size of the garage area beyond car space(s)
* `Has2ndFloor`: Boolean for having a substantial area on the second floor
* `TotalHalfBath`: Total number of half baths
* `HalfBathAbovePerc`: Percentage of half baths located above ground
* `TotalFullBath`: Total number of full baths
* `FullBathAbovePerc`: Percentage of full baths located above ground
* `OverallRating`: Product of `OverallQual` and `OverallCond`
* `ExterRating`: Product of ordinally encoded `ExterQual` and `ExterCond`
* `BsmtRating`: Product of ordinally encoded `BsmtQual` and `BsmtCond`
* `GarageRating`: Product of ordinally encoded `GarageQual` and `GarageCond`

### Transforming numerical features
* `date_to_age`: Changing year values to years ago (age) for `YearBuilt`, `YearRemodAdd`, `GarageYrBlt`
* `sqrt_normalized`: Changing area values to linear normalized measurements for `TotalLivArea`, `GarageArea`, `TotalBsmtSF`, `MasVnrArea`, `TotalPorchArea`

In [ ]:
# Function for properly naming new, transformed, and retained features
def add_cols_naming(transformer_name, feature_name):
    if (transformer_name in ['sqrt_normalized', 'date_to_age']):
        return transformer_name + '__' + feature_name
    if (transformer_name == 'keep') | (transformer_name == 'remainder'):
        return feature_name
    return transformer_name

# Function for creating a ratio with proper zeros
def add_ratio(X, nums, denoms):
    tmp = pd.DataFrame(X[nums].sum(axis=1)/X[denoms].sum(axis=1))
    return tmp.where(tmp.notna(), 0)

# Assumed car space to be 160 ft^2 (lower limit)
def GarageExtraArea(X):
    tmp = pd.DataFrame(X['GarageArea'] - X['GarageCars']*160)
    tmp = np.sqrt(tmp.where(tmp > 0, 0))
    return (tmp - tmp.mean())/tmp.std()

# Summary mark for BsmtFinType features
def BsmtFinCombined(X):
    tmp = pd.DataFrame((X['BsmtFinType1'] * X['BsmtFinSF1'] + X['BsmtFinType2'] * X['BsmtFinSF2']) /
                       (X['BsmtFinSF1'] + X['BsmtFinSF2']))
    return tmp.where(tmp.notna(), 0)

# Normalization of the square root of a feature
def sqrt_norm(X):
    tmp = np.sqrt(X)
    return (tmp - tmp.mean())/tmp.std()

# Convering year to years old from 2011
def date_to_age(X):
    tmp = 2011 - X
    return tmp.where(tmp != 2011, -1)

In [ ]:
# List of HouseStyle values indicating 2ndFloor
HouseStyle_2ndFloor = (['2Story', '2.5Fin', '2.5Unf'])

add_ct = ColumnTransformer(transformers=[
    ('BsmtFinCombined',
        FunctionTransformer(BsmtFinCombined),
        ['BsmtFinType1', 'BsmtFinType2', 'BsmtFinSF1', 'BsmtFinSF2']),
    ('OpenPorchPerc',
        FunctionTransformer(add_ratio, kw_args={'nums':['WoodDeckSF', 'OpenPorchSF'],
                                                'denoms':['TotalPorchArea']}),
        ['WoodDeckSF', 'OpenPorchSF', 'TotalPorchArea']),
    ('sqrt_norm_GarageExtraArea', FunctionTransformer(GarageExtraArea), ['GarageArea', 'GarageCars']),
    ('Has2ndFloor',
        FunctionTransformer(lambda x: pd.DataFrame((x['HouseStyle'].isin(HouseStyle_2ndFloor)) *
                                                   (x['2ndFlrSF'] > 100))),
        ['HouseStyle', '2ndFlrSF']),
    
    # Bath count features
    ('TotalHalfBath',
        FunctionTransformer(lambda x: pd.DataFrame(x['BsmtHalfBath'] + x['HalfBath'])),
        ['BsmtHalfBath', 'HalfBath']),
    ('HalfBathAbovePerc',
        FunctionTransformer(add_ratio, kw_args={'nums':['HalfBath'],
                                                'denoms':['BsmtHalfBath', 'HalfBath']}),
        ['BsmtHalfBath', 'HalfBath']),
    ('TotalFullBath',
        FunctionTransformer(lambda x: pd.DataFrame(x['BsmtFullBath'] + x['FullBath'])),
        ['BsmtFullBath', 'FullBath']),
    ('FullBathAbovePerc',
        FunctionTransformer(add_ratio, kw_args={'nums':['FullBath'],
                                                'denoms':['BsmtFullBath', 'FullBath']}),
        ['BsmtFullBath', 'FullBath']),
    
    # Combined ratings
    ('OverallRating',
        FunctionTransformer(lambda x: pd.DataFrame(np.sqrt(x['OverallQual'] * x['OverallCond']))),
        ['OverallQual', 'OverallCond']),
    ('ExterRating',
        FunctionTransformer(lambda x: pd.DataFrame(np.sqrt(x['ExterQual'] * x['ExterCond']))),
        ['ExterQual', 'ExterCond']),
    ('BsmtRating',
        FunctionTransformer(lambda x: pd.DataFrame(np.sqrt(x['BsmtQual'] * x['BsmtCond']))),
        ['BsmtQual', 'BsmtCond']),
    ('GarageRating',
        FunctionTransformer(lambda x: pd.DataFrame(np.sqrt(x['GarageQual'] * x['GarageCond']))),
        ['GarageQual', 'GarageCond']),

    # Numeric transformations
    ('date_to_age',
        FunctionTransformer(date_to_age),
        ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']),
    ('sqrt_normalized',
        FunctionTransformer(sqrt_norm),
        ['TotalLivArea', 'GarageArea', 'TotalBsmtSF', 'MasVnrArea', 'TotalPorchArea']),
    
    # Retaining some of the features involved in previous steps
    ('keep', 'passthrough', ['BsmtQual', 'HouseStyle', 'OverallQual', 'ExterQual', 'GarageQual'])
], remainder='passthrough', verbose_feature_names_out=add_cols_naming)

In [ ]:
# Mean encoding for descriptive features with many values
Mean_encoded_cols = ['HouseStyle', 'Neighborhood', 'MSSubClass']
Mean_encoder = TargetEncoder(target_type='continuous')

In [ ]:
# Orinal encoding for features with clear order logic
Ordinal_cols = ['BsmtExposure', 'HeatingQC', 'KitchenQual', 'Functional', 'FireplaceQu',
                'GarageFinish', 'PavedDrive', 'Fence', 'Electrical']

LotShape_cats = ['Reg','IR1','IR2','IR3']
BsmtExposure_cats = ['Gd', 'Av', 'Mn', 'No', 'NB']
Functional_cats = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
GarageFinish_cats = ['No', 'Unf', 'RFn', 'Fin']
PavedDrive_cats = ['Y', 'P', 'N']
Fence_cats = ['No', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
Electrical_cats = ['FuseP', 'FuseF', 'FuseA', 'SBrkr']

Order_encoder = OrdinalEncoder(categories=[
        BsmtExposure_cats, ExterQC_cats, ExterQC_cats, Functional_cats, BsmtQC_cats,
        GarageFinish_cats, PavedDrive_cats, Fence_cats, Electrical_cats],
    encoded_missing_value=-1)

In [ ]:
# One-hot encoding for the rest of features
OH_cols = ['MSZoning', 'LandContour', 'LotConfig', 'BldgType', 'RoofMatl', 'MasVnrType', 'Foundation',
           'CentralAir', 'Exterior1st', 'GarageType', 'SaleType', 'SaleCondition', 'LotShape']
OH_encoder = OneHotEncoder(drop='if_binary', sparse_output=False, handle_unknown='ignore')

In [ ]:
# Combined encoding ct
encode_ct = ColumnTransformer(transformers=[
    ('mean_encoded', Mean_encoder, Mean_encoded_cols),
    ('ordered', Order_encoder, Ordinal_cols),
    ('onehot', OH_encoder, OH_cols)
], remainder='passthrough')

In [ ]:
# Model setting
forest_model = RandomForestRegressor(n_estimators=375, max_depth=21, max_features=0.25, random_state=0)
# xgb_model = xgb.XGBRegressor(n_estimators=375, max_depth=21, learning_rate=0.05, early_stopping_rounds=5)

In [ ]:
# Processing pipeline without model for testing and analysis

# proc_pl = Pipeline(steps=[
#     ('imputed', impute_ct),
#     ('replaced_cats', replace_cats_ct),
#     ('added_cols', add_ct),
#     ('encoded', encode_ct)
# ])
# X_trans = proc_pl.fit_transform(X, y)

# X_trans['log'] = np.sqrt(X_trans['remainder__GarageExtraArea'])
# X_trans['norm'] = (X_trans['log'] - X_trans['log'].mean())/X_trans['log'].std()
# ax = sns.kdeplot(y)
# sns.histplot(X_trans.mean_encoded__HouseStyle, stat='density', color='r', ax=ax)

# X_melt = pd.melt(X_trans[['onehot__SaleType_New', 'onehot__SaleType_WD', 'onehot__SaleType_Oth', 'onehot__SaleType_COD']], var_name='SaleType', value_name='is')
# X_melt = X_melt[X_melt['is'] == 1]
# sns.boxplot(x=X_melt['SaleType'], y=y)
# X_trans['remainder__HasBsmt'].value_counts(dropna=False)
# HTML(X_trans.sort_values(by='remainder__LotFrontage').tail(10).to_html())

# HTML(pd.DataFrame(X_trans.corrwith(y).abs().sort_values(ascending=False)).to_html())
# HTML(pd.DataFrame(pd.Series(mutual_info_regression(X_trans, y), index=X_trans.columns).sort_values(ascending=False)).to_html())

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputed', ...), ('replaced_cats', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('BsmtExposure_prep', ...), ('BsmtFinType_prep', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.

In [ ]:
# Main pipeline
model_pl = Pipeline(steps=[
    ('imputed', impute_ct),
    ('replaced_cats', replace_cats_ct),
    ('added_cols', add_ct),
    ('encoded', encode_ct),
    ('model', forest_model)
])

model_pl.fit(X, y)

# XGB setup
# X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)
# proc_pl.fit(X_train, y_train)
# X_val_xgb = proc_pl.transform(X_val)
# model_pl.fit(X_train, y_train, model__eval_set=[(X_val_xgb, y_val)], model__verbose=False)

# MAE on training data
# print("MAE (Your appoach):")
# print((-1 * cross_val_score(model_pl, X, y, scoring='neg_mean_absolute_error')).mean())

In [ ]:
# Function for finding best parameter value for the model

# def score_per_param(param):
#     param_pl = Pipeline(steps=[
#         ('imputed', impute_ct),
#         ('replaced_cats', replace_cats_ct),
#         ('added_cols', add_ct),
#         ('encoded', encode_ct),
#         ('model', RandomForestRegressor(n_estimators=375, max_depth=21, max_features=param, random_state=0))
#     ])
#     return (-1 * cross_val_score(param_pl, X, y, scoring='neg_mean_absolute_error')).mean()
# {n : score_per_param(n) for n in np.arange(0.2, 0.5, 0.05)}

In [ ]:
# Preparing test data
X_test = test.copy()
X_test['TotalLivArea'] = X_test['TotalBsmtSF'] + X_test['GrLivArea']
X_test['TotalPorchArea'] = X_test['WoodDeckSF'] + X_test['OpenPorchSF'] + X_test['EnclosedPorch'] + X_test['3SsnPorch'] + X_test['ScreenPorch']

# Making prediction
pred_test = model_pl.predict(X_test)

d:\Program Files (x86)\Python\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 8, 10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [ ]:
# Creating output file
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': pred_test})
output.to_csv('submission.csv', index=False)